# RAG Evaluation challenge 0
### Develop an simple RAG application which gathers information from internet about the broad topics (user input) and uses the information as context for the LLM model and generate the output for the user query for same topic. Once the LLM genrates the response evaluate the response using the LLM and score it based on specified criterias.

## Planning
* Wide topic to be broken into smaller topics Ex: Topic : World's Leaders  -> List Important countries and their important leaders for past 100 years.
* Gether detailed information on the sub topics. Ex: Get the detailed information for about each leader on sertain topics like about life, education, family, struggles, contributions and awards and achievements.
*  Genretate test embaddings for the collected information and store them to vector store
* Genrate text embaddings for the user query and fetch the related documents from the vector store.
* Ingest the fetched docs and the user query to the LLM to generate the response.
* Evaluate the response generated by the LLM and score the response using LLM.
* if the confidence/score for the generated response is less then 60%, regenerate the response (maximum three ties)
* Reply to the user with the response and get the feedback.
* if the feed back is Positive or neutral, do nothing, else generate a new response considering the user feedback.



In [ ]:
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpointEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader, WikipediaLoader, wikipedia
from langchain_core.output_parsers import PydanticOutputParser
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate
from typing import List, Literal, Annotated, Optional
from pydantic import Field, BaseModel, AnyUrl
from langchain_core.runnables import RunnableSequence, RunnableBranch, RunnableLambda, RunnableParallel, RunnablePassthrough 
from dotenv import load_dotenv

In [2]:
# Loading the environement variables
load_dotenv()

True

In [3]:
# Describle Model output for a user topic
class Topic(BaseModel):
    regions : List[str] = Field(description="Sub-locations where topic has high presence/significance")
    names: dict[str, List[str]] = Field(description="Important list of names that works as examples for the topic region wise")
    details : List[str] =Field(description="Areas one should know about names")


In [4]:
# Prepare LLM Model
model_0 = ChatMistralAI(model_name='mistral-small-2603', temperature=0.4, timeout=60)
model_str_topic = model_0.with_structured_output(Topic) 

In [5]:
py_parser = PydanticOutputParser(pydantic_object= Topic )

In [6]:
sub_topic_promt = PromptTemplate(template='''You are a highly advance research bot who has deep understanding about {topic}. Break down the given topic   in to sub-topics based on the regions, also get the list of the details people should know about the sub-topics. Get the response in required output format. Generate output in format {format_instruction}''', 
                                 input_variables=['topic'],
                                 partial_variables={'format_instruction': py_parser.get_format_instructions()})

In [7]:
py_parser = PydanticOutputParser(pydantic_object= Topic )

In [8]:
topic = 'World Leaders from 20th century'

In [9]:
sub_topics_chain_0 = sub_topic_promt | model_0

In [10]:
sub_topics_0 = sub_topics_chain_0.invoke({'topic' : topic})

In [11]:
sub_topics_0.pretty_print()

================================== Ai Message ==================================

```json
{
  "regions": [
    "Europe",
    "North America",
    "Asia",
    "Africa",
    "South America",
    "Middle East",
    "Oceania"
  ],
  "names": {
    "Europe": [
      "Winston Churchill",
      "Charles de Gaulle",
      "Margaret Thatcher",
      "Konrad Adenauer",
      "Helmut Kohl"
    ],
    "North America": [
      "Franklin D. Roosevelt",
      "John F. Kennedy",
      "Lyndon B. Johnson",
      "Ronald Reagan",
      "Barack Obama"
    ],
    "Asia": [
      "Mahatma Gandhi",
      "Jawaharlal Nehru",
      "Deng Xiaoping",
      "Lee Kuan Yew",
      "Indira Gandhi"
    ],
    "Africa": [
      "Nelson Mandela",
      "Kwame Nkrumah",
      "Julius Nyerere",
      "Gamal Abdel Nasser",
      "Jomo Kenyatta"
    ],
    "South America": [
      "Getúlio Vargas",
      "Juan Perón",
      "Augusto Pinochet",
      "Hugo Chávez",
      "Luiz Inácio Lula da Silva"
    ],
    "Middle East"

In [12]:
sub_topics_chain_1 = sub_topic_promt | model_0 | py_parser

In [13]:
sub_topics_1 = sub_topics_chain_1.invoke({'topic' : topic})

In [14]:
print(sub_topics_1.regions, sub_topics_1.names, sub_topics_1.details, sep='\n')

['North America', 'Europe', 'Asia', 'Latin America', 'Africa', 'Middle East', 'Oceania']
{'North America': ['Franklin D. Roosevelt', 'Harry S. Truman', 'John F. Kennedy', 'Lyndon B. Johnson', 'Richard Nixon', 'Ronald Reagan'], 'Europe': ['Winston Churchill', 'Charles de Gaulle', 'Konrad Adenauer', 'Margaret Thatcher', 'Helmut Kohl', 'Francois Mitterrand'], 'Asia': ['Mahatma Gandhi', 'Jawaharlal Nehru', 'Mao Zedong', 'Ho Chi Minh', 'Lee Kuan Yew', 'Deng Xiaoping'], 'Latin America': ['Getúlio Vargas', 'Fidel Castro', 'Augusto Pinochet', 'Juan Perón', 'Salvador Allende'], 'Africa': ['Kwame Nkrumah', 'Jomo Kenyatta', 'Nelson Mandela', 'Gamal Abdel Nasser', 'Julius Nyerere'], 'Middle East': ['Gamal Abdel Nasser', 'Mohammad Reza Pahlavi', 'Anwar Sadat', 'Saddam Hussein', 'Golda Meir'], 'Oceania': ['Robert Menzies', 'Michael Somare', 'David Lange']}
['Key political figures of the 20th century', 'Leaders during major wars (e.g., World War II, Cold War)', 'Influential figures in decolonization 

## Step 2. Webscapping about the sub-topics(names)

In [15]:
import time
def wikipediaScapper(keyword : str, max_count=2 , dir=None ) -> List:
    '''Search wikipedia based on given keyword and store the articles to the given path.
        Input Parameters 
            keyword : Topic for the search
            max_count (Default=2) : Maximum document count 
            dir (Optional) : Directory to save data as text File 
        Output 
            str: Content for the keyword or file path 
    '''
    print(f'Looking Wikipedia for {keyword}')
    doc=None
    splitter = RecursiveCharacterTextSplitter(separators=['\n\n', '\n'], chunk_size = 200, chunk_overlap = 20)
    tries = 3
    while tries:
        try:
            wikiLoader = WikipediaLoader(query= keyword, load_max_docs=max_count)
            doc = wikiLoader.load_and_split(splitter)
            tries=0
        except:
            print('Ahh there seems some issue, Retrying...')
            time.sleep(1)
            tries-=1
    # print(doc)
    # docs = wikiLoader.aload()
    # if not dir:
    #     dir = '.' #Cureent working directory
    # dir = Path(dir)
    # file = dir / '{0}.txt'.format(keyword)
    # print('Writting file....')
    # with open(file, mode='w') as f:
    #     for doc in docs:
    #         f.write(doc.page_content)
    return doc



In [16]:
def getherDataOnTopics(topics: List[str]):
    '''Asyncronously gets the data from internet for the topics provided.
        input : 
            topics : list of topics to gether information for. 
        '''
    docs = []
    for topic in topics:
        print(f'Processing {topic}')
        doc = wikipediaScapper(keyword=topic, max_count=1)
        if doc:
            docs+=doc
    print('All topics processed...', f'Total Docs count {len(docs)}.' ) 
    return docs
    

In [17]:
sub_topics = set()
for regions, sub_topic in sub_topics_1.names.items(): 
    sub_topics = sub_topics.union(set(sub_topic[:2]))
sub_topics = list(sub_topics)


In [18]:
len(sub_topics)

14

In [19]:
docs = getherDataOnTopics(topics=sub_topics)
# print(len(routines))

Processing Fidel Castro
Looking Wikipedia for Fidel Castro
Ahh there seems some issue, Retrying...
Ahh there seems some issue, Retrying...
Ahh there seems some issue, Retrying...
Processing Robert Menzies
Looking Wikipedia for Robert Menzies
Ahh there seems some issue, Retrying...
Ahh there seems some issue, Retrying...
Ahh there seems some issue, Retrying...
Processing Michael Somare
Looking Wikipedia for Michael Somare
Processing Jawaharlal Nehru
Looking Wikipedia for Jawaharlal Nehru
Processing Getúlio Vargas
Looking Wikipedia for Getúlio Vargas
Processing Mahatma Gandhi
Looking Wikipedia for Mahatma Gandhi
Processing Charles de Gaulle
Looking Wikipedia for Charles de Gaulle
Processing Mohammad Reza Pahlavi
Looking Wikipedia for Mohammad Reza Pahlavi
Processing Franklin D. Roosevelt
Looking Wikipedia for Franklin D. Roosevelt
Ahh there seems some issue, Retrying...
Ahh there seems some issue, Retrying...
Ahh there seems some issue, Retrying...
Processing Jomo Kenyatta
Looking Wikipe

In [20]:
docs

[Document(metadata={'title': 'Michael Somare', 'summary': 'Sir Michael Thomas Somare  (9 April 1936 – 25 February 2021) was a Papua New Guinean politician. Widely called the "father of the nation" (Tok Pisin: Papa blo kantri), he was the first Prime Minister after independence. At the time of his death, Somare was also the longest-serving prime minister, having been in office for 17 years over three separate terms: from 1975 to 1980; from 1982 to 1985; and from 2002 to 2011. His political career spanned from 1968 until his retirement in 2017. Besides serving as PM, he was minister of foreign affairs, leader of the opposition and governor of East Sepik Province.\nHe served in a variety of positions. His base was not primarily in political parties but in East Sepik Province, the area that elected him. During his political career he was a member of the House of Assembly and after independence in 1975 the National parliament for the East Sepik Provincial – later open – seat. He was the fir

In [21]:
for doc in docs:
    if 'summary' in doc.metadata.keys():
        print('Removing summary...')
        doc.metadata.pop('summary')

Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...
Removing summary...


In [22]:
docs[0].metadata

{'title': 'Michael Somare',
 'source': 'https://en.wikipedia.org/wiki/Michael_Somare'}

In [23]:
#Embedding model
embed_model = HuggingFaceEmbeddings(model= "sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2723.01it/s]


In [24]:
vector_db = Chroma.from_documents(docs, embed_model)
vector_db

In [30]:
query = 'Who is Mahatma Gandhi?'

In [31]:
query_embedding = embed_model.embed_query(query)

In [32]:
result = vector_db.similarity_search_by_vector_with_relevance_scores(query_embedding, 3)

In [33]:
result

[(Document(id='93d6df0f-a296-4241-968d-ac01639f295f', metadata={'source': 'https://en.wikipedia.org/wiki/Mahatma_Gandhi', 'title': 'Mahatma Gandhi'}, page_content='Mohandas Karamchand Gandhi ( GA(H)N-dee; 2 October 1869 – 30 January 1948) was an Indian lawyer, anti-colonial nationalist and political ethicist who employed nonviolent resistance to lead the successful campaign for India\'s independence from British rule, and to later inspire movements for civil rights and freedom across the world. The honorific Mahātmā (Sanskrit: "great-souled", "venerable"), first applied to him in 1914 in South Africa, is now used throughout the world.'),
  0.5256630182266235),
 (Document(id='ed456c18-b4d0-4dd7-8641-988ead5cf7b6', metadata={'source': 'https://en.wikipedia.org/wiki/Mahatma_Gandhi', 'title': 'Mahatma Gandhi'}, page_content='\nGandhi\'s birthday, 2 October, is commemorated in India as Gandhi Jayanti, a national holiday, and worldwide as the International Day of Nonviolence. Gandhi is consi

In [62]:
context = ''
for doc in result:
    context+=doc[0].page_content
context

'Mohandas Karamchand Gandhi ( GA(H)N-dee; 2 October 1869 – 30 January 1948) was an Indian lawyer, anti-colonial nationalist and political ethicist who employed nonviolent resistance to lead the successful campaign for India\'s independence from British rule, and to later inspire movements for civil rights and freedom across the world. The honorific Mahātmā (Sanskrit: "great-souled", "venerable"), first applied to him in 1914 in South Africa, is now used throughout the world.\nGandhi\'s birthday, 2 October, is commemorated in India as Gandhi Jayanti, a national holiday, and worldwide as the International Day of Nonviolence. Gandhi is considered to be the Father of the Nation in post-colonial India. During India\'s nationalist movement and in several decades immediately after, he was also commonly called Bapu, an endearment roughly meaning "father".\nGandhi\'s father, Karamchand Uttamchand Gandhi (1822–1885), served as the dewan (chief minister) of Porbandar state. His family originated 

## Now as the vector store is prepared, get ready to feed the context to the RAG application so, it can retrieve the answer for the query.

In [77]:
resolve_query_prompt = PromptTemplate(template = 'Take help from the given text, try to answer the user query. Strictly Generate answer from within the content of the text else reply as "Context not helpful". Refer {context} to answer {query}..',
                                      input_variables=['context', 'query'])

In [78]:
query_resolve_chain = resolve_query_prompt | model_0 

In [79]:
query2 = 'Whan and where Mahatma Gandhi born and how did he die?'

In [75]:
query3 = 'Tell me about Nathhuram Godse?'

In [82]:
query4 ='Who Mahatma Gandhi died?'

In [83]:
response = query_resolve_chain.invoke({'context' : context,
                                     'query' : query4 })
response.pretty_print()

================================== Ai Message ==================================

Context not helpful.
